In [ ]:
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_curve, roc_auc_score, precision_recall_curve)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
train_df = pd.read_parquet("train_us.parquet")
test_df = pd.read_parquet("test_us.parquet")
MODEL_OUT = "autoencoder_model.pth"
benign_df = train_df[train_df['Attack']==0].drop(columns=['Attack'])

In [ ]:
len(train_df)

1919236

In [ ]:
SEED       = 42
BATCH_SIZE = 256
LR         = 1e-3
EPOCHS     = 100
VALID_RATIO = 0.1
PATIENCE   = 10
DROPOUT    = 0.2
NUM_WORKERS = 4
THRESHOLD_K = 3

USE_GPU = torch.cuda.is_available()
DEVICE  = torch.device("cuda" if USE_GPU else "cpu")

In [ ]:
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()


In [ ]:
import os
print(f"File size: {os.path.getsize('test_us.parquet') / (1024*1024):.2f} MB")

File size: 71.92 MB


In [ ]:
X_train,X_val = train_test_split(benign_df,test_size=0.1, random_state =42)

In [ ]:
X_test = test_df.drop(columns=['Attack'])
y_test = test_df['Attack'].values

NUM_FEATURES = X_train.shape[1]

print(f"Number of features: {NUM_FEATURES}")

Number of features: 69


In [ ]:
class AEDataset(Dataset):
    """Dataset for autoencoder — returns only X (target is X itself)."""
    def __init__(self, X):
        self.X = torch.tensor(X.values if hasattr(X, 'values') else X,
                              dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]


train_ds = AEDataset(X_train)
val_ds   = AEDataset(X_val)
test_ds  = AEDataset(X_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
class Autoencoder(nn.Module):
    """
    Symmetric autoencoder:
      Encoder: in_dim → 128 → 64 → 32 (bottleneck)
      Decoder: 32 → 64 → 128 → in_dim

    Sigmoid output because data is MinMaxScaled to [0, 1].
    """
    def __init__(self, in_dim, hidden1=128, hidden2=64, bottleneck=16, dropout=DROPOUT):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, bottleneck),
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, in_dim),
            nn.Sigmoid(),   # output bounded [0, 1] to match MinMaxScaler
        )

    def forward(self, x):
        if self.training:
          noise = torch.randn_like(x) * 0.1  # 10% Gaussian noise
          x_noisy = x + noise
          x_noisy = torch.clamp(x_noisy, 0, 1)  # keep in [0,1]
        else:
          x_noisy = x
        z = self.encoder(x_noisy)
        return self.decoder(z)


model = Autoencoder(NUM_FEATURES).to(DEVICE)
print(f"\nModel on: {DEVICE}")
print(model)


Model on: cuda
Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=69, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=64, out_features=16, bias=True)
    (9): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_f

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

history = {'train_loss': [], 'val_loss': []}


def train_one_epoch():
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        batch = batch.to(DEVICE)

        reconstruction = model(batch)
        loss = criterion(reconstruction, batch)  # target is the input itself

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch.size(0)
    return running_loss / len(train_loader.dataset)


def evaluate(loader):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            reconstruction = model(batch)
            loss = criterion(reconstruction, batch)
            running_loss += loss.item() * batch.size(0)
    return running_loss / len(loader.dataset)


print("\n" + "=" * 50)
print("TRAINING AUTOENCODER")
print("=" * 50)

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch()
    val_loss   = evaluate(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.6f} | "
          f"Val Loss: {val_loss:.6f} | "
          f"LR: {current_lr:.2e}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_OUT)
        print(f"  ✓ Saved best model (val_loss: {val_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (patience={PATIENCE})")
            break









TRAINING AUTOENCODER


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch   1/100 | Train Loss: 0.001921 | Val Loss: 0.000382 | LR: 1.00e-03
  ✓ Saved best model (val_loss: 0.000382)
Epoch   2/100 | Train Loss: 0.000724 | Val Loss: 0.000326 | LR: 1.00e-03
  ✓ Saved best model (val_loss: 0.000326)
Epoch   3/100 | Train Loss: 0.000693 | Val Loss: 0.000343 | LR: 1.00e-03
Epoch   4/100 | Train Loss: 0.000678 | Val Loss: 0.000326 | LR: 1.00e-03
  ✓ Saved best model (val_loss: 0.000326)
Epoch   5/100 | Train Loss: 0.000672 | Val Loss: 0.000330 | LR: 1.00e-03
Epoch   6/100 | Train Loss: 0.000664 | Val Loss: 0.000335 | LR: 1.00e-03
Epoch   7/100 | Train Loss: 0.000663 | Val Loss: 0.000317 | LR: 1.00e-03
  ✓ Saved best model (val_loss: 0.000317)
Epoch   8/100 | Train Loss: 0.000661 | Val Loss: 0.000351 | LR: 1.00e-03
Epoch   9/100 | Train Loss: 0.000659 | Val Loss: 0.000346 | LR: 1.00e-03
Epoch  10/100 | Train Loss: 0.000659 | Val Loss: 0.000343 | LR: 1.00e-03
Epoch  11/100 | Train Loss: 0.000658 | Val Loss: 0.000322 | LR: 5.00e-04
Epoch  12/100 | Train Loss: 0

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_OUT, weights_only=True))
print(f"\nLoaded best model from {MODEL_OUT}")



Loaded best model from autoencoder_model.pth


In [ ]:


def compute_reconstruction_errors(loader):
    """Compute per-sample MSE reconstruction errors."""
    model.eval()
    errors = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            reconstruction = model(batch)
            # Per-sample MSE: average across features, one error per sample
            mse = torch.mean((batch - reconstruction) ** 2, dim=1)
            max_err = torch.max((batch - reconstruction) ** 2, dim=1).values

            combined = 0.5 * (mse + max_err)
            errors.extend(combined.cpu().numpy())
    return np.array(errors)


val_errors = compute_reconstruction_errors(val_loader)

threshold_mean = val_errors.mean()
threshold_std  = val_errors.std()
threshold = threshold_mean +  threshold_std

# Also compute percentile-based thresholds for comparison
threshold_95 = np.percentile(val_errors, 95)
threshold_99 = np.percentile(val_errors, 99)

print(f"Benign validation errors — mean: {threshold_mean:.6f}, std: {threshold_std:.6f}")
print(f"Threshold (mean + {THRESHOLD_K}*std): {threshold:.6f}")
print(f"Threshold (95th percentile):  {threshold_95:.6f}")
print(f"Threshold (99th percentile):  {threshold_99:.6f}")

# Save thresholds
with open("threshold.txt", "w") as f:
    f.write(f"mean: {threshold_mean:.6f}\n")
    f.write(f"std: {threshold_std:.6f}\n")
    f.write(f"k: {THRESHOLD_K}\n")
    f.write(f"threshold (mean + k*std): {threshold:.6f}\n")
    f.write(f"threshold (95th percentile): {threshold_95:.6f}\n")
    f.write(f"threshold (99th percentile): {threshold_99:.6f}\n")
print(f"Saved thresholds to {'threshold.txt'}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Benign validation errors — mean: 0.003016, std: 0.015191
Threshold (mean + 3*std): 0.018207
Threshold (95th percentile):  0.011379
Threshold (99th percentile):  0.031063
Saved thresholds to threshold.txt


In [ ]:
test_errors = compute_reconstruction_errors(test_loader)

# Binary classification: error > threshold → anomaly (1), else benign (0)
predictions = (test_errors > threshold).astype(int)
true_labels = (y_test != 0).astype(int)  # 0=benign → 0, any attack → 1

# Overall binary report
report = classification_report(true_labels, predictions,
                               target_names=["Benign", "Anomaly"],
                               zero_division=0)
print("\nBinary Classification Report (Benign vs Anomaly):")
print(report)

accuracy = accuracy_score(true_labels, predictions)
print(f"Overall Accuracy: {accuracy:.4f}")

# Per-attack-type recall
print("\nPer-Attack-Type Recall:")
print("-" * 40)

attack_names = {0: "BENIGN", 1: "Bot", 2: "Brute Force", 3: "DDoS",
                4: "DoS", 5: "PortScan", 6: "Web Attack"}

per_attack_results = {}
for attack_id in sorted(test_df['Attack'].unique()):
    mask = y_test == attack_id
    name = attack_names.get(attack_id, f"Attack_{attack_id}")
    count = mask.sum()

    if attack_id == 0:
        # For benign: recall = what % correctly identified as benign (NOT anomaly)
        recall = 1 - predictions[mask].mean()
        print(f"  {name:15s} | Correctly identified: {recall:.4f} | Count: {count}")
    else:
        # For attacks: recall = what % flagged as anomaly
        recall = predictions[mask].mean()
        print(f"  {name:15s} | Recall: {recall:.4f} | Count: {count}")

    per_attack_results[name] = {'recall': recall, 'count': count}






Binary Classification Report (Benign vs Anomaly):
              precision    recall  f1-score   support

      Benign       0.75      0.93      0.83    395106
     Anomaly       0.67      0.31      0.42    178293

    accuracy                           0.74    573399
   macro avg       0.71      0.62      0.63    573399
weighted avg       0.72      0.74      0.70    573399

Overall Accuracy: 0.7378

Per-Attack-Type Recall:
----------------------------------------
  BENIGN          | Correctly identified: 0.9317 | Count: 395106
  Bot             | Recall: 0.0815 | Count: 981
  DDoS            | Recall: 0.5285 | Count: 98022
  PortScan        | Recall: 0.0384 | Count: 79290


In [ ]:

print("\n" + "=" * 50)
print("ROC AUC ANALYSIS")
print("=" * 50)

# ROC curve and AUC
fpr, tpr, thresholds_roc = roc_curve(true_labels, test_errors)
auc_score = roc_auc_score(true_labels, test_errors)
print(f"\nROC AUC Score: {auc_score:.4f}")

# F1-optimal threshold
precisions, recalls, thresholds_pr = precision_recall_curve(true_labels, test_errors)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds_pr[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]

print(f"\nF1-Optimal Threshold: {best_f1_threshold:.6f}")
print(f"  Precision: {precisions[best_f1_idx]:.4f}")
print(f"  Recall:    {recalls[best_f1_idx]:.4f}")
print(f"  F1-Score:  {best_f1:.4f}")

# Recall vs FPR tradeoff table
print("\nRecall vs False Positive Rate Tradeoff:")
print("-" * 50)
threshold_sweep = np.linspace(test_errors.max(), test_errors.min(), 10000)
for target_recall in [0.5, 0.6, 0.7, 0.8, 0.9]:
    for t in threshold_sweep:
        preds_t = (test_errors > t).astype(int)
        rec = preds_t[true_labels == 1].mean()
        if rec >= target_recall:
            fpr_t = preds_t[true_labels == 0].mean()
            print(f"  Recall {target_recall:.0%} → threshold: {t:.6f}, FPR: {fpr_t:.2%}")
            break
    else:
        print(f"  Recall {target_recall:.0%} → not achievable")

# Re-evaluate with F1-optimal threshold
print("\n" + "=" * 50)
print(f"EVALUATION WITH F1-OPTIMAL THRESHOLD ({best_f1_threshold:.6f})")
print("=" * 50)

predictions_optimal = (test_errors > best_f1_threshold).astype(int)
report_optimal = classification_report(true_labels, predictions_optimal,
                                       target_names=["Benign", "Anomaly"],
                                       zero_division=0)
print(report_optimal)

per_attack_results_optimal = {}
print("Per-Attack-Type Recall (F1-Optimal Threshold):")
print("-" * 40)
for attack_id in sorted(test_df['Attack'].unique()):
    mask = y_test == attack_id
    name = attack_names.get(attack_id, f"Attack_{attack_id}")
    count = mask.sum()
    if attack_id == 0:
        recall_opt = 1 - predictions_optimal[mask].mean()
        print(f"  {name:15s} | Correctly identified: {recall_opt:.4f} | Count: {count}")
    else:
        recall_opt = predictions_optimal[mask].mean()
        print(f"  {name:15s} | Recall: {recall_opt:.4f} | Count: {count}")
        per_attack_results_optimal[name] = {'recall': recall_opt, 'count': count}


# Save text report (using F1-optimal threshold)
with open("classification_reports.txt", "w") as f:
    f.write(f"Binary Classification Report (F1-Optimal Threshold: {best_f1_threshold:.6f})\n")
    f.write("=" * 50 + "\n")
    f.write(report_optimal + "\n")
    f.write(f"ROC AUC: {auc_score:.4f}\n\n")
    f.write("Per-Attack-Type Recall (F1-Optimal Threshold)\n")
    f.write("=" * 50 + "\n")
    for name, res in per_attack_results_optimal.items():
        f.write(f"  {name:15s} | Recall: {res['recall']:.4f} | Count: {res['count']}\n")
print(f"\nSaved reports to {'classification_reports.txt'}")


# Save AUC and optimal threshold info
with open("threshold.txt", "a") as f:
    f.write(f"\nROC AUC: {auc_score:.6f}\n")
    f.write(f"F1-optimal threshold: {best_f1_threshold:.6f}\n")
    f.write(f"F1-optimal F1-score: {best_f1:.4f}\n")
    f.write(f"F1-optimal precision: {precisions[best_f1_idx]:.4f}\n")
    f.write(f"F1-optimal recall: {recalls[best_f1_idx]:.4f}\n")



ROC AUC ANALYSIS

ROC AUC Score: 0.7801

F1-Optimal Threshold: 0.001068
  Precision: 0.5203
  Recall:    0.8902
  F1-Score:  0.6567

Recall vs False Positive Rate Tradeoff:
--------------------------------------------------
  Recall 50% → threshold: 0.003391, FPR: 20.09%
  Recall 60% → threshold: 0.001760, FPR: 29.29%
  Recall 70% → threshold: 0.001192, FPR: 35.45%
  Recall 80% → threshold: 0.001050, FPR: 37.33%
  Recall 90% → threshold: 0.000838, FPR: 40.93%

EVALUATION WITH F1-OPTIMAL THRESHOLD (0.001068)
              precision    recall  f1-score   support

      Benign       0.93      0.63      0.75    395106
     Anomaly       0.52      0.89      0.66    178293

    accuracy                           0.71    573399
   macro avg       0.72      0.76      0.70    573399
weighted avg       0.80      0.71      0.72    573399

Per-Attack-Type Recall (F1-Optimal Threshold):
----------------------------------------
  BENIGN          | Correctly identified: 0.6296 | Count: 395106
  Bot 

In [ ]:
def plot_loss(history):
    """Plots training and validation loss across epochs."""
    plt.figure(figsize=(10, 6))
    epochs_ran = len(history['train_loss'])
    epochs_range = range(1, epochs_ran + 1)

    plt.plot(epochs_range, history['train_loss'], label='Training Loss',
             marker='o', linestyle='--', markersize=3)
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss',
             marker='o', markersize=3)

    plt.title('Autoencoder — Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    out_path = "loss_plot.png"
    plt.savefig(out_path)
    print(f"Saved loss plot to: {out_path}")
    plt.close()


## Reconstruction Error Distribution
def plot_error_distribution(test_errors, y_test, threshold, optimal_threshold):
    """Plots reconstruction error histograms: benign vs attacks with both thresholds."""
    plt.figure(figsize=(12, 7))

    benign_errors = test_errors[y_test == 0]
    attack_errors = test_errors[y_test != 0]

    plt.hist(benign_errors, bins=100, alpha=0.6, label='Benign', color='steelblue', density=True)
    plt.hist(attack_errors, bins=100, alpha=0.6, label='Attacks', color='crimson', density=True)
    plt.axvline(threshold, color='black', linestyle='--', linewidth=2,
                label=f'Threshold k={THRESHOLD_K} ({threshold:.4f})')
    plt.axvline(optimal_threshold, color='green', linestyle='--', linewidth=2,
                label=f'F1-Optimal ({optimal_threshold:.4f})')

    plt.title('Reconstruction Error Distribution — Benign vs Attacks')
    plt.xlabel('Reconstruction Error')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    out_path = "reconstruction_error_distribution.png"
    plt.savefig(out_path)
    print(f"Saved error distribution plot to: {out_path}")
    plt.close()


## Per-Attack Recall Bar Chart
def plot_per_attack_recall(per_attack_results):
    """Bar chart of recall per attack type."""
    names = [k for k in per_attack_results if k != "BENIGN"]
    recalls = [per_attack_results[k]['recall'] for k in names]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(names, recalls, color='steelblue', edgecolor='black')

    for bar, val in zip(bars, recalls):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha='center', va='bottom', fontsize=10)

    plt.title('Autoencoder — Per-Attack-Type Recall')
    plt.xlabel('Attack Type')
    plt.ylabel('Recall')
    plt.ylim(0, 1.15)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()

    out_path ="per_attack_recall.png"
    plt.savefig(out_path)
    print(f"Saved per-attack recall plot to: {out_path}")
    plt.close()


## ROC Curve
def plot_roc_curve(fpr, tpr, auc_score):
    """Plots the ROC curve with AUC score."""
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='steelblue', linewidth=2,
             label=f'Autoencoder (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')

    plt.title('ROC Curve \u2014 Autoencoder Anomaly Detection')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate (Attack Recall)')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    out_path = "roc_curve.png"
    plt.savefig(out_path)
    print(f"Saved ROC curve to: {out_path}")
    plt.close()


## Precision-Recall Curve
def plot_precision_recall_curve(precisions, recalls, best_f1, best_f1_idx):
    """Plots the Precision-Recall curve with F1-optimal point marked."""
    plt.figure(figsize=(8, 6))
    plt.plot(recalls, precisions, color='steelblue', linewidth=2, label='Autoencoder')
    plt.scatter(recalls[best_f1_idx], precisions[best_f1_idx],
                color='red', s=100, zorder=5,
                label=f'F1-Optimal (F1={best_f1:.4f})')

    plt.title('Precision-Recall Curve \u2014 Autoencoder Anomaly Detection')
    plt.xlabel('Recall (Attack Detection Rate)')
    plt.ylabel('Precision')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    out_path = "precision_recall_curve.png"
    plt.savefig(out_path)
    print(f"Saved precision-recall curve to: {out_path}")
    plt.close()


plot_loss(history)
plot_error_distribution(test_errors, y_test, threshold, best_f1_threshold)
plot_per_attack_recall(per_attack_results_optimal)
plot_roc_curve(fpr, tpr, auc_score)
plot_precision_recall_curve(precisions, recalls, best_f1, best_f1_idx)

print("\n" + "=" * 50)
print("DONE — All outputs saved to 'models/autoencoder/'")
print("=" * 50)

Saved loss plot to: loss_plot.png
Saved error distribution plot to: reconstruction_error_distribution.png
Saved per-attack recall plot to: per_attack_recall.png
Saved ROC curve to: roc_curve.png
Saved precision-recall curve to: precision_recall_curve.png

DONE — All outputs saved to 'models/autoencoder/'
